# Colombia Seismic Analysis Dashboard

## Objetivo

Analizar la actividad sísmica registrada en Colombia mediante técnicas de análisis exploratorio de datos (EDA) y visualización interactiva en Power BI, con el fin de identificar patrones espaciales, temporales y geológicos relacionados con la ocurrencia de eventos sísmicos.

## Fuente de Datos

Datos obtenidos del Servicio Geológico Colombiano (SGC).

Fuente oficial:
[Catálogo de Sismos del SGC](https://www.sgc.gov.co/catalogo)

Los datos originales no se distribuyen en este repositorio y deben descargarse directamente desde la fuente oficial.

## Conjunto de Datos descargado cuya fuente es el SGS tiene las siguientes caracteristicas:

- Registros: 407067
- Variables: 18
- intervalo: 1/06/1993 hasta 16/11/2026

## Alcance del Proyecto

Este proyecto se centra en:

- Analizar la distribución geográfica de los eventos sísmicos en Colombia.
- Identificar las zonas con mayor actividad sísmica.
- Evaluar la distribución de magnitudes y profundidades.
- Analizar la evolución temporal de la actividad sísmica.
- Explorar la relación entre magnitud y profundidad.
- Construir un dashboard geoespacial e interactivo en Power BI para apoyar la interpretación de los datos.

## Descripción General de los Datos

El conjunto de datos contiene registros históricos de eventos sísmicos localizados dentro del territorio colombiano y zonas cercanas, recopilados y publicados por el Servicio Geológico Colombiano (SGC).

Cada registro corresponde a un evento sísmico e incluye información relacionada con:

- Ubicación geográfica (latitud y longitud).
- Fecha y hora del evento.
- Magnitud del sismo.
- Profundidad del evento.
- Tipo de magnitud calculada.
- Agencia responsable del reporte.

## Fuente de Datos

Datos obtenidos del Servicio Geológico Colombiano (SGC).

# Estructura del Proyecto

1. Importación de librerías
2. Limpieza de datos
3. Resumen de la evaluación inicial del conjunto de datos limpio
4. Ventana temporal de análisis e inicio del EDA
5. Preguntas de negocio a resolver

   5.1. ¿Dónde ocurre la mayor actividad sísmica en Colombia?

   5.2. ¿Cómo ha evolucionado la actividad sísmica a lo largo del tiempo?

   5.3. ¿Cuáles son las magnitudes más frecuentes?

   5.4.¿Cómo se distribuyen las profundidades de los eventos?

   5.5.¿Existe relación entre magnitud y profundidad?

   5.6.¿Qué departamentos o regiones presentan mayor actividad sísmica?

   5.7.¿Qué agencias registran mayor volumen de eventos?

6. Hallazgos principales
7. Limitaciones del análisis
8. Exportación del conjunto de datos para Power BI




- ¿Dónde ocurren más sismos en Colombia?

- ¿Cómo ha evolucionado la actividad sísmica a través del tiempo?

- ¿Cuáles son las magnitudes más frecuentes?

- ¿Cómo se distribuyen las profundidades?

- ¿Existe relación entre magnitud y profundidad?

- ¿Qué departamentos presentan mayor actividad sísmica?

# 1. Importación de librerías

In [1]:
# ==================================================
# 📚 IMPORTACIÓN DE LIBRERÍAS
# ==================================================

# 📂🧹🔍📊 Librerías básicas para exploración y análisis

# Librería para trabajar con DataFrames
import pandas as pd
pd.set_option('display.max_columns', None) #Parametrizacion que permite ver todas las columnas

# Librería para carpetas
import os

# Librería para cálculos numéricos y matrices
import numpy as np

# Librerías para visualización
import matplotlib.pyplot as plt      # Más personalizable
import seaborn as sns                # Más moderna y estética
import folium
from folium.plugins import HeatMap   # Mapas de Calor


# ==================================================
# 🤖 MACHINE LEARNING - SCIKIT LEARN
# ==================================================

# 📈 Modelos de regresión
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

# 🔗 Pipeline (cadena de transformaciones)
from sklearn.pipeline import Pipeline

# ⚖️ Escalado de variables
from sklearn.preprocessing import StandardScaler

# 📏 Métricas de evaluación
from sklearn.metrics import mean_squared_error, r2_score

# ✂️ División de datos
from sklearn.model_selection import train_test_split

# ✅ Validación cruzada
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict

# 🛡️ Regresión Ridge para reducir el sobreajuste
from sklearn.linear_model import Ridge

# 🔍 Búsqueda de hiperparámetros
from sklearn.model_selection import GridSearchCV

In [2]:
# Carga y copia del conjunto de datos

df = pd.read_csv('../Data/Raw/20261116_205614_2ed2cc4d-5104-4dc5-a7ed-57b9acb89685.csv') 			#read_csv Modulo para cargar archivos, se carga sin títulos,
										                                #el Header = none, hace que no se tome la primera fila para tomar titulo

df_cl = df.copy()							                           	#Modulo para copia de datos

In [3]:
df_cl.info()

<class 'pandas.DataFrame'>
RangeIndex: 407067 entries, 0 to 407066
Data columns (total 18 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   id               407067 non-null  str    
 1   status           407067 non-null  str    
 2   rms              407064 non-null  float64
 3   agency           407067 non-null  str    
 4   gap              406872 non-null  float64
 5   place            407067 non-null  str    
 6   local_time       407067 non-null  str    
 7   utc_time         407067 non-null  str    
 8   magnitude        407067 non-null  float64
 9   magnitude_error  221710 non-null  float64
 10  nst              407065 non-null  float64
 11  mag_type         407067 non-null  str    
 12  latitude         407067 non-null  float64
 13  latitude_error   406802 non-null  float64
 14  longitude        407067 non-null  float64
 15  longitude_error  406802 non-null  float64
 16  depth            407067 non-null  int64  
 17  de

In [4]:
#Link de descarga https://www.sgc.gov.co/catalogo, en un rango de fechas del 1 de junio de 1993 hasta 16-11-2026

# 2. Limpieza de datos

In [5]:
"""Se procede a realizar el borrado de las siguientes columnas, dado que no aportan
 valor al alcance del presente proyecto; status, latitude_error, longitude_error, depth_error, 
 magnitude_error, rms, nst y gap. Tambien se elimina la columna utc_time para que solo quede 
 una unica varaible de fecha la cual sera local_time """


df_cl = df_cl.drop(columns=["status", "latitude_error", "longitude_error", "depth_error",
                          "magnitude_error", "rms", "nst", "gap", "utc_time"])


"""Se procede a realizar las siguiente conversiones de datos"""

'Se procede a realizar las siguiente conversiones de datos'

In [6]:
#Conversion de datos

df_cl["agency"] = df_cl["agency"].astype("category")                                    #Se convierte las siguietnes variables en categoricas.
df_cl["local_time"] = pd.to_datetime(df_cl["local_time"],format="%Y-%m-%d %H:%M:%S")    #Se da formato de fecha a la columna e tiempo.

Se crean intervalos para clasificar las magnitudes de los sismos

| Magnitud | Clasificación |
|-----------|-----------|
| < 2.0 | Micro sismo |
| 2.0 - 2.9 | Muy menor |
| 3.0 - 3.9 | Menor |
| 4.0 - 4.9 | Ligero |
| 5.0 - 5.9 | Moderado |
| 6.0 - 6.9 | Fuerte |
| 7.0 - 7.9 | Mayor |
| ≥ 8.0 | Gran terremoto |

In [7]:
# Cracion de los bins segun la tabla anterior

bins = [-2, 2, 3, 4, 5, 6, 7, 8, 10]				#Se genera los intervalos de los bins

labels = [
    "Micro sismo",
    "Muy menor",
    "Menor",
    "Ligero",
    "Moderado",
    "Fuerte",
    "Mayor",
    "Gran terremoto"
]								#Se crea las Categorías de los Bins

df_cl["magnitude_class"] = pd.cut(				#Se crea la Columna categoría con los bins

    df_cl["magnitude"],
    bins=bins,
    labels=labels
)

In [8]:
df_cl.info()

<class 'pandas.DataFrame'>
RangeIndex: 407067 entries, 0 to 407066
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   id               407067 non-null  str           
 1   agency           407067 non-null  category      
 2   place            407067 non-null  str           
 3   local_time       407067 non-null  datetime64[us]
 4   magnitude        407067 non-null  float64       
 5   mag_type         407067 non-null  str           
 6   latitude         407067 non-null  float64       
 7   longitude        407067 non-null  float64       
 8   depth            407067 non-null  int64         
 9   magnitude_class  407067 non-null  category      
dtypes: category(2), datetime64[us](1), float64(3), int64(1), str(3)
memory usage: 44.1 MB


# 3. Resumen de la evaluación inicial del conjunto de datos limpio

El conjunto de datos limpio contiene:

407.067 registros
10 variables
Todas las variables fueron revisadas y clasificadas de acuerdo con su relevancia para el negocio, calidad de los datos y utilidad analítica.

El diccionario de datos detallado fue documentado por separado.

### Resumen de la clasificación de variables

| Categoría | Número de Variables |
|----------|----------|
| Variables alfanumericas | 3 |
| Campos de Fecha | 2 |
| Variables Categóricas | 1 |
| Variables Booleanas | 0 |
| Métricas Operativas | 4 |


### Principales observaciones

- La información disponible cubre más de 30 años de actividad sísmica registrada en Colombia.
- El conjunto de datos no presenta problemas significativos de completitud en las variables seleccionadas para el análisis.
- Las variables de ubicación, magnitud, profundidad y tiempo constituyen los principales ejes analíticos del proyecto.

### Documentación de referencia

Se realizó una revisión detallada de las 18 variables orignales del conjunto de datos. El diccionario de datos completo fue documentado por separado y se encuentra disponible dentro de la carpeta Data/Processed archivo excel "Diccionario de datos.xlsx" como soporte metodológico para las decisiones de limpieza, transformación y análisis realizadas en este proyecto

# 4. Ventana temporal de análisis es inicio del EDA

El conjunto de datos abarca desde 1 de junio de 1993 hasta 16-11-2026, por lo cual se utilizara la totalidad de la fuente.

El objetivo es usar todos los datos disponibles para determiar de manera mas precisa patrones de tiempo y lugar para diferentes graficos

# 5. Preguntas de negocio a resolver

# 5.1. ¿Dónde ocurre la mayor actividad sísmica en Colombia?

In [9]:
df_cl["place"].value_counts(normalize= False)

place
Los Santos - Santander, Colombia    116707
Zapatoca - Santander, Colombia       21978
Cepitá - Santander, Colombia         13573
Villanueva - Santander, Colombia      7999
Dabeiba - Antioquia, Colombia         7135
                                     ...  
Gualmatán - Nariño, Colombia             1
Mitú - Vaupés, Colombia                  1
Venecia - Cundinamarca, Colombia         1
Miraflores - Guaviare, Colombia          1
L                                        1
Name: count, Length: 1603, dtype: int64

In [10]:
import folium
mapa = folium.Map(
    location=[4.5, -74],
    zoom_start=5
)

HeatMap(
    df_cl[["latitude", "longitude"]].values.tolist(),
    radius=10
).add_to(mapa)

mapa

1. Lo que logramos encontrar en el intervalo de tiempo realizado es que la region de los santos santander, es la region con mas sismos en colombia
2. otros punto piripiri - Meta

### Hallazgo preliminar: Concentración espacial de eventos sísmicos

La exploración geoespacial evidencia una concentración importante de eventos sísmicos en la región de Los Santos (Santander), zona ampliamente reconocida por su actividad sísmica.

Adicionalmente, se identificó una concentración relevante de eventos en la región de Piripiri (Meta), convirtiéndose en uno de los principales focos sísmicos observados dentro del conjunto de datos analizado.

| Magnitud | Clasificación |
|-----------|-----------|
| < 2.0 | Micro sismo |
| 2.0 - 2.9 | Muy menor |
| 3.0 - 3.9 | Menor |
| 4.0 - 4.9 | Ligero |
| 5.0 - 5.9 | Moderado |
| 6.0 - 6.9 | Fuerte |
| 7.0 - 7.9 | Mayor |
| ≥ 8.0 | Gran terremoto |

In [11]:
# consulta sector de coordenadas
Latitud_up = 7.0
Latitud_down = 6.5
Longitud_right = -72.9
Longitud_left = -73.4

df_sector = df_cl[(df_cl["latitude"]  <= Latitud_up) & (df_cl["latitude"] >= Latitud_down)
                   & (df_cl["longitude"] <= Longitud_right) & (df_cl["longitude"] >= Longitud_left)]

mapa = folium.Map(
    location=[8, -74],
    zoom_start=5
)

HeatMap(
    df_sector[["latitude", "longitude"]].values.tolist(),
    radius=10
).add_to(mapa)

mapa

In [12]:
#tipos de sismos en el punto de mas generacion

df_sector["magnitude_class"].value_counts(normalize=True) * 100

magnitude_class
Micro sismo       61.922191
Muy menor         31.256426
Menor              6.053579
Ligero             0.710805
Moderado           0.054763
Fuerte             0.002235
Mayor              0.000000
Gran terremoto     0.000000
Name: proportion, dtype: float64

In [13]:
#Como se logra evidencia a pesar de que el sector de los santos es el que mas presenta sismos, 
#Hitoricamente los sismos son de magnitud media para abajo, solo el 0.002235% de los sismos ha sido de magnitud fuerte o entre  6 y 6.9

# 5.2. ¿Cómo ha evolucionado la actividad sísmica a lo largo del tiempo?

# 6. Hallazgos Principales

# 7. Limitaciones del Análisis

# 8. Exportación del conjunto de datos para Power BI